# Creating Geoparquet files from stac Geojsons 

This notebook is created for collecting stac geojson files generated by libraries such as `sar-pipeline` and converting them to geoparquet files that could be uploaded to AWS S3 or stored locally.

The generates `.parquet` can then be loaded later to perform spatio-temporal queries and operations.

The idea behind this notebook is to create a parquet catalog of all stac products every time new products are available as a single file that could be queries without the need for an stac api.

Note that an **Item** here means a single spatio-temporal item, such as one observation in a dataset.

## Setup
### Import required libraries

In [1]:
from sar_pipeline.utils.stac import (
    read_stac_items_from_s3,
    create_and_upload_geoparquet,
)

### Environment setup

The setups here will appear as input boxes for you to provide the input for the requested parameter; otherwise, it will be set to the default value specified below.

The inputs are:

* `bucket`: S3 bucket name where the stac items are stored.
* `prefix`: Prefix for the directory where the stac items are stored on S3.
* `item suffixes`: List of the file name / s3 path suffixes to use for querying the stac items.
* `product name`: This will be used as the output name for the geoparquet file.
* `AWS profile`: AWS profile name for interacting with AWS S3. AWS profile is needed if you are using different profile than the `default` profile. AWS access id, secret key and aws access token if required should be configured via the credential file or AWS SSO.
* `num to read`: Number of items to read. Good for sampling.

In [2]:
bucket = input("Enter S3 bucket name: ") or "dea-public-data-dev"
prefix = input("Enter S3 prefix: ") or "experimental"
item_suffixes = input("Enter item suffixes (comma-separated): ") or "stac-item.json"
item_suffixes = [s.strip() for s in item_suffixes.split(",")]
product_name = input("Enter product name for output file: ") or "ga_s1_nrb"
aws_profile = input("Enter AWS profile name: ")
if aws_profile == "":
    aws_profile = None
num_to_read = input("Enter number of items to read: ")
if num_to_read:
    num_to_read = int(num_to_read)
else:
    num_to_read = None

### Reading the item directly from AWS S3

The function below takes the input above and queries and reads the stac items from s3 directly into a list of pystac items.

In [3]:
loaded_items = read_stac_items_from_s3(
    bucket=bucket,
    prefix=prefix,
    item_suffixes=item_suffixes,
    aws_profile=aws_profile,
    num_to_read=num_to_read,
)

INFO:botocore.tokens:Loading cached SSO token for dev
INFO:botocore.tokens:SSO Token refresh succeeded
Reading STAC items from S3: 100%|██████████| 20/20 [00:02<00:00,  9.84it/s, Last read=experimental/baseline/c1_testing/ga_s1_nrb_iw_hh_1/t055_117599_iw3/2015/05/25/20150525T120610/ga_s1a_nrb_0-1-0_T055-117599-IW3_20150525T120610Z_stac-item.json]


### Set the S3 output path

In [4]:
parquet_filename = f"{product_name}.parquet"
output_s3_path = f"s3://{bucket}/{prefix}/{parquet_filename}"
print(f"Writing to {output_s3_path}...")

Writing to s3://dea-public-data-dev/experimental/ga_s1_nrb.parquet...


### Creating and uploading the geoparquet file

The function below takes the list of pystac items loaded in the previous step and converts them to a single `.parquet` file and if provided, writes it to the `output_path`.
The output path could be a S3 url, in which case, the function will upload the parquet file to the provided S3 path.

In [5]:
table = create_and_upload_geoparquet(
    items=loaded_items,
    output_path=output_s3_path,
    aws_profile=aws_profile,
)

INFO:botocore.tokens:Loading cached SSO token for dev


Uploading to S3 bucket: dea-public-data-dev, key: experimental/ga_s1_nrb.parquet
